# Quantum Horizon — Bitcoin & Ethereum Quantum-Threat Assessment
**ArXivist-generated reproduction notebook**

Paper: [arXiv:2606.14484](https://arxiv.org/abs/2606.14484)
Generated: 2026-07-21

This notebook walks through all six quantitative models in the paper: the
systemic break-year Monte-Carlo forecast, the mining-competitiveness model,
the Bitcoin and Ethereum exposure models, the mempool-race model, the
migration-race (Mosca's inequality) model, and the top-20 readiness survey.
No GPU is needed — everything here is lightweight NumPy/SciPy/pandas
computation.

In [ ]:
# Environment check -- no GPU needed
import sys
print(f"Python: {sys.version}")

import numpy as np
print(f"NumPy: {np.__version__}")
import pandas as pd
print(f"pandas: {pd.__version__}")


In [ ]:
# Install the project in editable mode (run once)
import subprocess
result = subprocess.run(["pip", "install", "-e", ".."], capture_output=True, text=True)
if result.returncode != 0 and "externally-managed-environment" in result.stderr:
    result = subprocess.run(["pip", "install", "--break-system-packages", "-e", ".."], capture_output=True, text=True)
print(result.stdout[-2000:] if result.returncode == 0 else result.stderr[-2000:])


## Paper Overview

**Core distinction**: two quantum algorithms matter, and conflating them is
the paper's identified "single most common error" in this debate.

- **Shor's algorithm** breaks ECDSA (secp256k1) and BLS (BLS12-381)
  signatures — the real threat, since it can derive a private key from a
  public key.
- **Grover's algorithm** only quadratically speeds up brute-force search —
  not a meaningful threat to proof-of-work mining, which is additionally
  protected by fault-tolerant per-operation cost, a $\sqrt{K}$
  parallelization wall, and difficulty adjustment.

**Six quantitative models**, built to cross-check each other (the paper's
own rule: "a model is credited with confirming another only insofar as it
shares no inputs, assumptions, or method with it"):
1. Systemic break-year Monte-Carlo forecast (§3.2)
2. Mining-competitiveness model (§2.2)
3. Bitcoin exposure model (§4)
4. Mempool-race model (§4.3)
5. Ethereum exposure model (§5)
6. Migration-race / Mosca's-inequality model (§7.4)

Plus a qualitative top-20 cryptocurrency readiness survey (§6) that the
paper itself says is "a sourced field assessment, not a model."


## Model 1: Mining competitiveness — Grover vs. proof-of-work

Bitcoin's June-2026 difficulty implies ~$2^{79}$ classical hashes per block.
Grover needs only ~$2^{40}$ iterations (~$\sqrt{2^{79}}$), but each
iteration is a fault-tolerant, reversible double-SHA-256 oracle of 474,168
T-gates. Effective hashrate scales linearly with gate speed, calibrated to
the Aggarwal et al. 2017 benchmark (13.8 GH/s @ 66.7 MHz).

In [ ]:
from quantum_horizon.mining import MiningCompetitivenessModel

model = MiningCompetitivenessModel()
hashrate_100ghz = model.effective_hashrate(100)
print(f"Effective hashrate @ 100 GHz: {hashrate_100ghz:.2f} TH/s  (paper: ~21 TH/s)")
print(f"One modern ASIC: ~200 TH/s -- a single quantum machine is roughly "
      f"{200/hashrate_100ghz:.0f}x slower")

# Grover parallelization wall: K machines get only sqrt(K) speedup
for k in [1, 100, 10000, 1000000]:
    parallel = model.parallel_hashrate(100, k)
    print(f"  {k:>10,} machines -> {parallel:.2e} TH/s combined (sqrt(K) scaling, not linear)")


## Model 2: Systemic break-year Monte-Carlo forecast

The paper's headline result: a **bimodal** distribution blending a
bottom-up physics/hardware-scaling estimator (mode ~2052) and an
expert-survey estimator (mode ~2038-2040) at equal weight.

In [ ]:
from quantum_horizon.timeline import PhysicsBasedEstimator, SurveyBasedEstimator, SystemicForecastModel

physics = PhysicsBasedEstimator()
survey = SurveyBasedEstimator()
forecast = SystemicForecastModel(physics, survey)

result = forecast.run(n_draws=100000, survey_weight=0.5, seed=0)

print(f"Median break-year: {result['median']:.1f}  (paper: ~2046-2047)")
print(f"P(CRQC by 2035): {result['cdf_2035']:.1%}  (paper: ~1-in-6, ~16.7%)")
print(f"P(CRQC by 2040): {result['cdf_2040']:.1%}  (paper: ~30%)")
print(f"P(CRQC by 2050): {result['cdf_2050']:.1%}  (paper: ~60%)")
print(f"80% range: {result['range_80pct'][0]:.0f}-{result['range_80pct'][1]:.0f}  (paper: ~2032-2060)")


## Model 3: Bitcoin exposure — irreducible vs. migratable vs. protected

Reconciles three independent 2025-26 on-chain measurements into the paper's
decomposition (Figure 3): only ~2.3M of Bitcoin's ~6M exposed coins are
*irreducibly* at risk (dormant, lost, or Satoshi-era).

In [ ]:
from quantum_horizon.exposure import BitcoinExposureModel

btc = BitcoinExposureModel()

sources = {"glassnode": 0.302, "coinbase": 6.9/19.4, "coindesk": 7.0/19.4, "deloitte_2020": 0.25}
reconciled = btc.reconcile_sources(sources)
print(f"Reconciled exposed fraction: {reconciled:.1%}  (paper: ~30%)")

decomposition = btc.decompose(total_supply_btc_millions=19.4, irreducible_btc_millions=2.3, migratable_btc_millions=3.7)
for k, v in decomposition.items():
    print(f"  {k}: {v:.3f}")


## Model 4: Mempool-race — the one live attack window

Even "protected" (fresh-address) Bitcoin has one moment of vulnerability:
when spent, the public key briefly sits in the mempool before confirming. A
**fast-clock** CRQC (superconducting/photonic) might derive the key and
broadcast a replacement transaction in time; a **slow-clock** machine
(trapped-ion/neutral-atom) cannot.

In [ ]:
from quantum_horizon.attacks import MempoolRaceModel

mp = MempoolRaceModel()
best_case = mp.snipe_success_probability(derivation_minutes=9, propagation_delay_minutes=0, confirmations_required=1)
realistic = mp.snipe_success_probability(derivation_minutes=10.5, propagation_delay_minutes=2.0, confirmations_required=1)

print(f"Best-case success probability: {best_case:.1%}  (paper: 41%)")
print(f"Realistic success probability: {realistic:.1%}  (paper: ~30%)")
print(f"Fast-clock (superconducting) feasible: {mp.is_feasible('superconducting')}")
print(f"Slow-clock (trapped-ion) feasible: {mp.is_feasible('trapped-ion')}  (paper: impossible)")


## Model 5: Ethereum exposure — broader but more migratable

Ethereum's account model means every account that has ever sent a
transaction has an effectively public key. Cross-checked via a top-down
composition and a bottom-up on-chain build.

In [ ]:
from quantum_horizon.exposure import EthereumExposureModel

eth = EthereumExposureModel()
top_down = eth.top_down_estimate(staked_fraction=0.32, contract_fraction=0.08)
bottom_up = eth.bottom_up_estimate(naive_beacon_overcount_fraction=0.21, correction_factor=2.3)
low, high = eth.reconcile(top_down, bottom_up)

print(f"Top-down estimate: {top_down:.1%}  (paper: ~55-63%)")
print(f"Bottom-up estimate: {bottom_up:.1%}  (paper: ~45-55%)")
print(f"Reconciled range: {low:.1%} - {high:.1%}  (paper: 50-65%, most defensibly 55-60%)")


## Model 6: Migration race — Mosca's inequality

The decision rule: assets are at risk whenever
(time-to-start-migrating + time-to-migrate) > time-until-CRQC-arrives.
The paper sweeps three start-time scenarios against three CRQC estimates.

In [ ]:
from quantum_horizon.migration import MigrationRaceModel

mr = MigrationRaceModel()
scenarios = {
    "prompt": {"start_year": 2026, "migration_duration_years": 4},
    "delayed": {"start_year": 2030, "migration_duration_years": 5},
    "severe_delay": {"start_year": 2033, "migration_duration_years": 5},
}
crqc_estimates = {"aggressive": 2035, "survey_median": 2040, "conservative": 2050}

df = mr.run_scenarios(scenarios, crqc_estimates)
print(df.to_string(index=False))
print()
at_risk = df[df["at_risk"]]
print(f"At-risk combinations: {len(at_risk)}/{len(df)}")
print("(Paper: 'the only at-risk case in the entire sweep is a severely delayed start... running against an early machine')")


## Top-20 readiness survey (qualitative, not a model)

Transcribed directly from the paper's Table 3 -- the paper itself states
this is "a sourced field assessment, not a model."


In [ ]:
from quantum_horizon.survey import QuantumReadinessSurvey

survey = QuantumReadinessSurvey()
ratings = survey.load_ratings("../data/table3_readiness_ratings.csv")
stats = survey.summary_stats(ratings)

print(ratings[["coin", "rating"]].sort_values("rating", ascending=False).to_string(index=False))
print(f"\nNo coin reaches 5: {stats['no_coin_reaches_5']}  (paper: 'none is fully post-quantum')")


## Paper's Reported Results (for reference)

The numbers below are transcribed directly from the paper's Table 1 and
body text, for comparison against your own runs.

In [ ]:
paper_results = {
    "Logical qubits to break secp256k1 (2026 frontier)": "1,200-1,450",
    "Physical qubits to break secp256k1 (2026 frontier)": "<500,000",
    "Best demonstrated physical qubits (2026)": "~1,000-1,200",
    "P(CRQC by 2035)": "~1-in-6 (8-24% across weightings)",
    "P(CRQC by 2040)": "~30%",
    "P(CRQC by 2050)": "~60%",
    "Bitcoin exposed at rest": "~6.0M BTC (~30% of supply)",
    "Bitcoin irreducibly at risk": "~2.3M BTC (~12%)",
    "Ethereum at used accounts": "50-65% of supply, most defensibly 55-60%",
    "Quantum mining hashrate @ 100GHz": "~21 TH/s (~1/10th of one ASIC)",
    "Migration race": "prompt 2026 start finishes ~2029-2031, beats even an optimistic 2035 CRQC",
}
for k, v in paper_results.items():
    print(f"  {k}: {v}")

print()
print("To reproduce all figures and headline numbers:")
print("  python run_all.py --config configs/config.yaml --output-dir results/")


## What to do next

1. **Full run with all figures**: `python run_all.py --config configs/config.yaml --output-dir results/`
2. **Sensitivity sweep on the timeline forecast**: `python run_timeline_forecast.py --config configs/config.yaml --sensitivity-sweep`

**Top 3 things to be aware of** (see `README.md` and `sir.json` for the full list):

1. This paper's SIR confidence (0.64) is below the usual 0.65 comfort
   threshold -- several sub-models (Monte-Carlo distributional form,
   mempool-race timing, exposure-reconciliation method) are described
   narratively in the paper without explicit formulas.
2. The timeline forecast's distributional parameters were *tuned* to match
   the paper's reported percentiles, not independently derived -- see
   `timeline/survey_estimator.py` and `physics_estimator.py` docstrings.
3. The mining model's "machines needed for 51%" figure is ~6x off from the
   paper's stated value (4.5e14 vs ~7e13) -- an open, documented
   discrepancy, not silently forced to match.
